# Lab 6b: Spark on Databricks

Run the cells one at a time. The flights are in `{datalake_path}/bronze/flights`


## Start Spark

In [0]:
# Parámetros base para que todo quede organizado en DBFS
base_path = "/Volumes/mycatalog262/myschema262/myvolume262"
raw_path = f"{base_path}/raw"
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"
ml_path = f"{base_path}/ml"

for p in [base_path, raw_path, bronze_path, silver_path, gold_path, ml_path]:
    dbutils.fs.mkdirs(p)

display(dbutils.fs.ls(bronze_path))

## Read the flights

The schema is given explicitly.

In [0]:
schema = (
    'flight_id long, flight_date date, airline_code string, '
    'flight_number string, tail_number string, origin string, '
    'destination string, scheduled_departure timestamp, '
    'actual_departure timestamp, distance_km int, cancelled int')

flights = spark.read.csv(f"{bronze_path}/flights/",
                         header=True,
                         schema=schema)
flights.printSchema()

## An action: count the flights

In [0]:
flights.count()

## Transformations are lazy

These lines only note the plan. Open the Jobs tab of the web UI: no new job appears.

In [0]:
from pyspark.sql import functions as F
flown = (flights.filter(F.col('cancelled') == 0).withColumn(
    'delay_min', (F.col('actual_departure').cast('long') -
                  F.col('scheduled_departure').cast('long')) / 60))
late = flown.filter(F.col('delay_min') > 15)

An action runs the whole plan.

In [0]:
late.count()

## Partitions, stages and the shuffle

Grouping by origin needs a shuffle. The plan shows it as an `Exchange`.

In [0]:
by_origin = flights.groupBy('origin').count()
by_origin.explain()
by_origin.orderBy(F.desc('count')).show(5)

## The same query in Spark SQL

In [0]:
flights.createOrReplaceTempView('vuelos')
by_origin_sql = spark.sql(
    'SELECT origin, count(*) AS count FROM vuelos GROUP BY origin')
by_origin_sql.explain()
by_origin_sql.orderBy(F.desc('count')).show(5)

## A join with a small table

The airports table is small, so Spark can send a copy of it to every executor instead of moving the flights.

In [0]:
airports = spark.read.csv(f"{bronze_path}/airports",
                          header=True,
                          inferSchema=True)
airports.printSchema()


In [0]:
airports.show(5)

In [0]:
by_country = (flights.join(
    airports, flights.origin == airports.airport_code).groupBy(
        'country').count().orderBy(F.desc('count')))
by_country.explain()
by_country.show()

## A window function: running total of delay minutes per airline

In [0]:
from pyspark.sql import Window
daily = flown.groupBy('airline_code', 'flight_date').agg(
    F.sum('delay_min').alias('delay_min_total'))
running = Window.partitionBy('airline_code').orderBy('flight_date')
total = daily.withColumn('running_delay_min',
                         F.sum('delay_min_total').over(running))
total.filter(F.col('airline_code') == 'AND').orderBy('flight_date').show(5)

## Spark and pandas on the same calculation

Both read all the files and compute the average delay of each airline.

In [0]:
import time

start = time.time()
spark_result = (flown.groupBy('airline_code').agg(
    F.avg('delay_min').alias('avg_delay_min')).orderBy(
        'airline_code').toPandas())
print(f'Spark: {time.time() - start:.1f} s')
spark_result

In [0]:
import glob

import pandas as pd

start = time.time()
columns = [
    'airline_code', 'scheduled_departure', 'actual_departure', 'cancelled'
]
frames = [
    pd.read_csv(path, usecols=columns, parse_dates=['scheduled_departure', 'actual_departure'])
    for path in sorted(glob.glob(f"{bronze_path}/flights/*.csv"))
]
data = pd.concat(frames)
data = data[data['cancelled'] == 0]
data['delay_min'] = (data['actual_departure'] -
                     data['scheduled_departure']).dt.total_seconds() / 60
pandas_result = data.groupby('airline_code')['delay_min'].mean()
print(f'pandas: {time.time() - start:.1f} s')
pandas_result

## Stop Spark

In [0]:
spark.stop()